In [ ]:
import pandas as pd
import sys, os
sys.path.insert(0, os.path.join("..", "daily_ingestion"))
from config import HOSPITAL_TO_REGION, REGION_POPULATIONS

### Count all hospitals per region (from config.py HOSPITAL_TO_REGION)

In [ ]:
df_all = pd.DataFrame([
    {"Hospital": h, "Region": r}
    for h, r in HOSPITAL_TO_REGION.items()
])

all_counts = df_all.groupby("Region").size().reset_index(name="NHospitals_All")
all_counts

### Filter to ED-only hospitals (exclude specialist units that don't report trolley data)

Specialist/non-ED facilities: National Orthopaedic Hospital Cappagh, National Rehabilitation Hospital, South Infirmary Victoria University Hospital, St. Luke's Radiation Oncology Network, St. Michael's Hospital, St. Columcille's Hospital

In [ ]:
# These are specialist units — no ED trolley reporting
NON_ED = {
    "National Orthopaedic Hospital Cappagh",
    "National Rehabilitation Hospital",
    "South Infirmary Victoria University Hospital",
    "St. Luke's Radiation Oncology Network",
    "St. Michael's Hospital",
    "St. Columcille's Hospital",
}

df_ed = df_all[~df_all["Hospital"].isin(NON_ED)].copy()

print(f"Total hospitals: {len(df_all)}")
print(f"Excluded (non-ED): {len(NON_ED)}")
print(f"ED hospitals: {len(df_ed)}")

ed_counts = df_ed.groupby("Region").size().reset_index(name="NHospitals_ED")
ed_counts

### Scale to per 10k population (matches trolley data units)

In [ ]:
df_pop = pd.read_csv("../data/encatchment_areas.csv")
dict_pop = dict(zip(df_pop.iloc[:, 0], df_pop.iloc[:, 1]))

df = ed_counts.copy()
df["Population"] = df["Region"].map(dict_pop)
df["EDHospitalsPer10k"] = (df["NHospitals_ED"] / df["Population"]) * 10000

df

### Export for Model V5

In [ ]:
df[["Region", "EDHospitalsPer10k"]].to_csv("../data/hospitals_per_region.csv", index=False)
print("Saved to ../data/hospitals_per_region.csv")
df[["Region", "EDHospitalsPer10k"]]